# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10` (Ablation Study A1 -> A4)


In [ ]:
def patch_environment():
    """Patches missing triton.ops, language attributes, and handles CPU/GPU fallbacks for Kaggle and Colab."""
    import os
    import sys
    import types
    import importlib.machinery

    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 1. Patch Triton mock safely
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__spec__ = importlib.machinery.ModuleSpec("triton", None)
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    # Ensure triton.language exists to satisfy transformers.modeling_utils import checks
    if not hasattr(m_triton, "language"):
        m_lang = types.ModuleType("triton.language")
        m_lang.__spec__ = importlib.machinery.ModuleSpec("triton.language", None)
        m_lang.constexpr = lambda x: x
        setattr(m_triton, "language", m_lang)
        sys.modules["triton.language"] = m_lang

    if not hasattr(m_triton, "cdiv"):
        setattr(m_triton, "cdiv", lambda a, b: (a + b - 1) // b)
    if not hasattr(m_triton, "jit"):
        setattr(m_triton, "jit", lambda *a, **k: (lambda f: f))

    m_ops = types.ModuleType("triton.ops")
    m_ops.__spec__ = importlib.machinery.ModuleSpec("triton.ops", None)
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.__spec__ = importlib.machinery.ModuleSpec("triton.ops.matmul_perf_model", None)
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 2. Bulletproof fallback for bitsandbytes
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes", None)
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")
        m_bnb_nn.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn", None)

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb_nn.modules.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn.modules", None)
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.47.0"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️ CẢNH BÁO: BẠN CHƯA BẬT GPU!')
    print('👉 Vui lòng vào menu Runtime -> Change runtime type -> Chọn T4 GPU (hoặc Dual T4 trên Kaggle) -> Nhấn Save và chạy lại từ đầu.')


In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


In [ ]:
# BƯỚC 3: HUẤN LUYỆN CROSS-ENCODER RERANKER (RECIPE CoREB: LoRA r=16, alpha=16, nDCG@10)
import os, sys

py_bin = sys.executable
print('\n' + '='*80 + '\n🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES\n' + '='*80)
mod = "training_lab.training.reranker.train_reranker_lora" if os.path.exists("training_lab") else "training.reranker.train_reranker_lora"
dpath = "training_lab/data/triplets/massive_reranker_pairs.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_reranker_pairs.jsonl"
!{py_bin} -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
# ==============================================================================
# BƯỚC 4: HUẤN LUYỆN EMBEDDING (RECIPE MoAI: 1 EPOCH CHUẨN VÀNG SOTA, TỰ ĐỒNG BỘ DATA)
# ==============================================================================
import os, sys, time

# 0. Tự động kiểm tra và clone mã nguồn & dữ liệu nếu phiên Kaggle vừa khởi động lại
dpath = None
for cand in [
    "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "/kaggle/working/Research-assist/training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "data/triplets/massive_embedding_triplets.jsonl"
]:
    if os.path.exists(cand):
        dpath = cand
        break

if not dpath:
    print("⏳ Phiên Kaggle mới chưa có dữ liệu, đang tự động clone repository từ GitHub...", flush=True)
    !rm -rf /kaggle/working/temp_repo
    !git clone https://github.com/nguyentatmanh/Research-assist.git /kaggle/working/temp_repo
    !cd /kaggle/working/temp_repo && (git checkout upgrade 2>/dev/null || true)
    !cp -rn /kaggle/working/temp_repo/training_lab /kaggle/working/
    !rm -rf /kaggle/working/temp_repo
    dpath = "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl"
    print("✅ Đã đồng bộ thành công dữ liệu massive_embedding_triplets.jsonl!")

# 1. Cập nhật thư viện tương thích tuyệt đối
!pip install -q 'accelerate>=1.1.0' --upgrade sentence-transformers

# 2. Khởi chạy huấn luyện 1 Epoch chuẩn (~625 steps, hoàn tất trong ~2.5 tiếng)
py_bin = sys.executable
print('\n' + '='*80 + '\n⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING 1 EPOCH CHUẨN VÀNG SOTA (~2.5 TIẾNG)\n' + '='*80)
if os.path.exists("/kaggle/working/training_lab"):
    os.chdir("/kaggle/working")
    mod = "training_lab.training.embedding.train_embedding_lora"
elif os.path.exists("training_lab"):
    mod = "training_lab.training.embedding.train_embedding_lora"
else:
    mod = "training.embedding.train_embedding_lora"

!{py_bin} -m {mod} --epochs 1 --lr 3e-5 --data {dpath} --batch-size 8


In [ ]:
# ==============================================================================
# BƯỚC 5: HUẤN LUYỆN AGENT 2 (DOCUMENT READER) TRÊN TẬP DOCLAYNET THẬT (10 NHÃN)
# ==============================================================================
import os, sys

print('\n' + '='*80 + '\n📄 [STAGE 3] HUẤN LUYỆN LAYOUTLMv3 FOCAL LOSS TRÊN BẢNG & CÔNG THỨC THẬT\n' + '='*80)
target = "training_lab/agent2_reader/train_agent2_t4x2.py" if os.path.exists("training_lab") else "agent2_reader/train_agent2_t4x2.py"
!{sys.executable} -u {target} --epochs 3


In [ ]:
import sys
# BƯỚC 6: CHẠY ABLATION STUDY THỰC NGHIỆM ĐỘC LẬP (A1 -> A4)
import os
print('\n' + '='*80 + '\n🏆 [STAGE 4] CHẠY ABLATION STUDY ĐỐI CHỨNG THEO CHUẨN TURKISH LEGAL RAG\n' + '='*80)
mod = "training_lab.evaluation.run_ablation_benchmark" if os.path.exists("training_lab") else "evaluation.run_ablation_benchmark"
!{sys.executable} -m {mod}


In [ ]:
import sys
# ==============================================================================
# BƯỚC 7: ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD, RGB BENCHMARK & TỰ SỬA SAI)
# Đánh giá 4 chỉ số cấp thế giới: Negative Rejection, Noise Robustness, Groundedness, Latency
# ==============================================================================
import os
bm_script = "training_lab/evaluation/world_class_benchmark.py" if os.path.exists("training_lab") else "evaluation/world_class_benchmark.py"
!{sys.executable} {bm_script}


In [ ]:
# ==============================================================================
# BƯỚC 8: TỰ ĐỘNG ĐÓNG GÓI TOÀN BỘ TRỌNG SỐ VỀ TAB KAGGLE OUTPUT & FILE ZIP
# ==============================================================================
import os, shutil
from pathlib import Path

base_out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
BUNDLE_DIR = base_out / 'AI20_Trained_Adapters'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

search_dirs = [Path('/kaggle/working'), Path('training_lab'), Path('.')]
found_weights = set()

# Quét cả adapter_model (LoRA) và model (full head / LayoutLMv3)
patterns = ['**/adapter_model.safetensors', '**/adapter_model.bin', '**/model.safetensors', '**/pytorch_model.bin']
for sdir in search_dirs:
    if sdir.exists():
        for pat in patterns:
            for w in sdir.glob(pat):
                if 'checkpoint' in str(w).lower() or 'AI20_Trained_Adapters' in str(w):
                    continue
                found_weights.add(w.resolve())

print(f'🔍 Tìm thấy {len(found_weights)} tệp trọng số hợp lệ.')

for w in sorted(found_weights):
    parent = w.parent
    # Xác định tên agent từ đường dẫn thư mục
    if parent.name == "weights":
        agent_name = parent.parent.name
    elif parent.parent.name in ["weights", "adapters"]:
        agent_name = f"{parent.parent.name}_{parent.name}"
    else:
        agent_name = parent.name

    dest = BUNDLE_DIR / agent_name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(w, dest / w.name)
    
    # Sao chép toàn bộ config và tokenizer metadata
    for ext in ['*.json', '*.txt', '*.model']:
        for cfg_f in parent.glob(ext):
            shutil.copy2(cfg_f, dest / cfg_f.name)
    print(f'📦 Đã gom hoàn chỉnh: {agent_name} -> {dest}')

OUT_ZIP = base_out / 'AI20_MultiAgent_Weights_DualT4.zip'
shutil.make_archive(str(OUT_ZIP).replace('.zip', ''), 'zip', str(BUNDLE_DIR))
size_mb = round(OUT_ZIP.stat().st_size / (1024*1024), 2) if OUT_ZIP.exists() else 0

print('=' * 80)
print(f'🎉 [THÀNH CÔNG] File zip đã sẵn sàng: {OUT_ZIP} ({size_mb} MB)')
print('👉 Tải ngay tại tab Output bên phải Kaggle hoặc dùng link 1-click ở Bước 9!')
print('=' * 80)


In [ ]:
# ==============================================================================
# BƯỚC 9: TỰ ĐỘNG TẢI FILE ZIP LÊN GOOGLE DRIVE (HOẶC TẢI VỀ 1-CLICK)
# ==============================================================================
import os, shutil
from pathlib import Path

OUT_ZIP = Path("/kaggle/working/AI20_MultiAgent_Weights_DualT4.zip") if Path("/kaggle/working/AI20_MultiAgent_Weights_DualT4.zip").exists() else Path("./outputs/AI20_MultiAgent_Weights_DualT4.zip")
TARGET_FOLDER_ID = "YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE"  # <-- Điền ID thư mục Google Drive của bạn vào đây

print('=' * 80)
print('🚀 [STAGE 9] XUẤT FILE TRỌNG SỐ RA GOOGLE DRIVE / MÁY TÍNH')
size_mb = round(OUT_ZIP.stat().st_size / (1024*1024), 2) if OUT_ZIP.exists() else 0
print(f'📦 Tệp xuất bản: {OUT_ZIP.name} (Dung lượng: {size_mb} MB)')
print('=' * 80)

# LỰA CHỌN 1: TẠO LINK TẢI 1-CLICK TRỰC TIẾP VỀ MÁY TÍNH (NHANH NHẤT TRÊN KAGGLE)
try:
    from IPython.display import FileLink, display
    print('\n📥 [LỰA CHỌN 1] Bấm vào link màu xanh bên dưới để tải trực tiếp file zip về máy tính:')
    # Lưu ý: Kaggle proxy yêu cầu relative path để không bị lỗi 404
    rel_path = OUT_ZIP.name if os.path.exists('/kaggle/working') else str(OUT_ZIP)
    display(FileLink(rel_path))
except Exception as e:
    print(f'ℹ️ FileLink: {e}')

# LỰA CHỌN 2: NẾU ĐANG CHẠY TRÊN GOOGLE COLAB
if os.path.exists('/content/drive/MyDrive'):
    dest_drive = Path(f'/content/drive/MyDrive/AI20_MultiAgent_Weights_DualT4.zip')
    shutil.copy2(OUT_ZIP, dest_drive)
    print(f'\n🎉 [GOOGLE COLAB] Đã sao chép thành công vào Drive: {dest_drive}')

# LỰA CHỌN 3: TẢI TRỰC TIẾP LÊN THƯ MỤC GOOGLE DRIVE QUA PYDRIVE2
print(f'\n☁️ [LỰA CHỌN 3] Tải trực tiếp vào thư mục Google Drive: https://drive.google.com/drive/folders/{TARGET_FOLDER_ID}')
try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pydrive2"], check=False)
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive

    gauth = GoogleAuth()
    gauth.CommandLineAuth() # Click vào link hiện ra để cấp quyền Google và dán mã xác thực
    drive = GoogleDrive(gauth)

    file_metadata = {
        'title': OUT_ZIP.name,
        'parents': [{'id': TARGET_FOLDER_ID}]
    }
    gfile = drive.CreateFile(file_metadata)
    gfile.SetContentFile(str(OUT_ZIP))
    print(f'⏳ Đang tải {OUT_ZIP.name} lên Google Drive...')
    gfile.Upload()
    print(f'🎉 [THÀNH CÔNG] Đã tải hoàn tất file zip vào thư mục Google Drive của bạn!')
    print(f'👉 Link kiểm tra: https://drive.google.com/drive/folders/{TARGET_FOLDER_ID}')
except Exception as e:
    print(f'\nℹ️ Ghi chú: Nếu chưa cấu hình OAuth Google, bạn hãy dùng [LỰA CHỌN 1] để tải file zip về máy trong 1 click, sau đó kéo thả vào link Drive trên nhé! ({e})')
